In [1]:
!pip -q install timm albumentations opencv-python-headless

In [2]:
import os
import re
import cv2
import time
import json
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings("ignore")

In [3]:
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 0

PAYUTCH_ROOT = "/kaggle/input/datasets/payutch/fall-video-dataset"

# IMPORTANT: use your real Kaggle path here
PRETRAINED_PATH = "/kaggle/input/datasets/sadiamostafa/improved/best_payutch_improved_model.pth"

IMG_SIZE = 192
CLIP_LEN = 20
BATCH_SIZE = 2
EPOCHS_TUNE = 4

BACKBONE_NAME = "tf_efficientnetv2_b0.in1k"
GRU_HIDDEN = 256
GRU_LAYERS = 1
DROPOUT = 0.30

LR_HEAD = 3e-4
LR_BACKBONE = 3e-5
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
GRAD_CLIP = 1.0

USE_AMP = True
NUM_CLASSES = 2
CLASS_NAMES = ["non_fall", "fall"]
CLASS_TO_ID = {"non_fall": 0, "fall": 1}

N_SPLITS = 5
VAL_FOLD = 0
TEST_FOLD = 1

SAVE_DIR = "/kaggle/working/payutch_final_refine"
os.makedirs(SAVE_DIR, exist_ok=True)

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".mpeg", ".mpg"}

print("DEVICE:", DEVICE)
print("PRETRAINED_PATH:", PRETRAINED_PATH)
print("SAVE_DIR:", SAVE_DIR)

DEVICE: cuda
PRETRAINED_PATH: /kaggle/input/datasets/sadiamostafa/improved/best_payutch_improved_model.pth
SAVE_DIR: /kaggle/working/payutch_final_refine


In [4]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)

In [5]:
def natural_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', str(s))]

def is_video_file(path):
    return Path(path).suffix.lower() in VIDEO_EXTS

In [6]:
rows = []

for label_name, subfolder in [("fall", "Fall/Raw_Video"), ("non_fall", "No_Fall/Raw_Video")]:
    folder = Path(PAYUTCH_ROOT) / subfolder
    if not folder.exists():
        print("Missing:", folder)
        continue

    for fn in sorted(os.listdir(folder), key=natural_key):
        full = str(folder / fn)
        if not is_video_file(full):
            continue

        stem = Path(fn).stem.lower()

        rows.append({
            "path": full,
            "label_name": label_name,
            "label": CLASS_TO_ID[label_name],
            "stem": stem
        })

pay_meta = pd.DataFrame(rows)

print("Payutch raw total:", len(pay_meta))
print(pay_meta["label_name"].value_counts())
pay_meta.head()

Payutch raw total: 6988
label_name
non_fall    3848
fall        3140
Name: count, dtype: int64


,path,label_name,label,stem
0,/kaggle/input/datasets/payutch/fall-video-data...,fall,1,20240912_101331
1,/kaggle/input/datasets/payutch/fall-video-data...,fall,1,20240912_101427
2,/kaggle/input/datasets/payutch/fall-video-data...,fall,1,20240912_101520
3,/kaggle/input/datasets/payutch/fall-video-data...,fall,1,20240912_101626
4,/kaggle/input/datasets/payutch/fall-video-data...,fall,1,20240912_101723


In [7]:
stem_label_counts = pay_meta.groupby("stem")["label"].nunique()
conflict_stems = stem_label_counts[stem_label_counts > 1].index.tolist()

print("Conflicting stems found:", len(conflict_stems))

if len(conflict_stems) > 0:
    before = len(pay_meta)
    pay_meta = pay_meta[~pay_meta["stem"].isin(conflict_stems)].reset_index(drop=True)
    print("Removed rows:", before - len(pay_meta))

print("After cleanup:", len(pay_meta))
print(pay_meta["label_name"].value_counts())

Conflicting stems found: 1892
Removed rows: 3784
After cleanup: 3204
label_name
non_fall    1956
fall        1248
Name: count, dtype: int64


In [8]:
pay_meta["group"] = pay_meta["stem"]

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
splits = list(sgkf.split(pay_meta, y=pay_meta["label"], groups=pay_meta["group"]))

train_val_idx, test_idx = splits[TEST_FOLD]
pay_train_val_df = pay_meta.iloc[train_val_idx].reset_index(drop=True)
pay_test_df = pay_meta.iloc[test_idx].reset_index(drop=True)

sgkf2 = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED + 7)
inner = list(sgkf2.split(pay_train_val_df, y=pay_train_val_df["label"], groups=pay_train_val_df["group"]))

train_idx, val_idx = inner[VAL_FOLD]
pay_train_df = pay_train_val_df.iloc[train_idx].reset_index(drop=True)
pay_val_df = pay_train_val_df.iloc[val_idx].reset_index(drop=True)

print("Train:", len(pay_train_df))
print("Val:", len(pay_val_df))
print("Test:", len(pay_test_df))

print("\nTrain counts:")
print(pay_train_df["label_name"].value_counts())

print("\nVal counts:")
print(pay_val_df["label_name"].value_counts())

print("\nTest counts:")
print(pay_test_df["label_name"].value_counts())

Train: 2050
Val: 513
Test: 641

Train counts:
label_name
non_fall    1249
fall         801
Name: count, dtype: int64

Val counts:
label_name
non_fall    312
fall        201
Name: count, dtype: int64

Test counts:
label_name
non_fall    395
fall        246
Name: count, dtype: int64


In [9]:
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.35),
    A.HueSaturationValue(p=0.20),
    A.GaussianBlur(blur_limit=(3, 5), p=0.10),
    A.MotionBlur(blur_limit=(3, 5), p=0.08),
    A.Normalize(),
    ToTensorV2(),
])

valid_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(),
    ToTensorV2(),
])

In [10]:
def sample_indices(total_frames, clip_len=20, train_mode=True):
    total_frames = max(int(total_frames), 1)

    if total_frames <= clip_len:
        return np.linspace(0, total_frames - 1, clip_len).astype(int).tolist()

    if train_mode:
        start = random.randint(0, total_frames - clip_len)
        return list(range(start, start + clip_len))

    return np.linspace(0, total_frames - 1, clip_len).astype(int).tolist()

def load_clip_from_video(path, clip_len=20, train_mode=True):
    cap = cv2.VideoCapture(path)

    if not cap.isOpened():
        cap.release()
        raise ValueError(f"Could not open video: {path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise ValueError(f"Invalid frame count: {path}")

    idxs = sample_indices(total_frames, clip_len=clip_len, train_mode=train_mode)
    idx_set = set(idxs)

    frames = {}
    fid = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if fid in idx_set:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames[fid] = frame

        fid += 1
        if len(frames) == len(idx_set):
            break

    cap.release()

    if len(frames) == 0:
        raise ValueError(f"No frames extracted: {path}")

    ordered = []
    first = next(iter(frames.values()))
    last = None

    for i in idxs:
        if i in frames:
            last = frames[i]
        if last is None:
            last = first
        ordered.append(last)

    return ordered

In [11]:
class PayutchDataset(Dataset):
    def __init__(self, df, transform=None, clip_len=20, train_mode=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.clip_len = clip_len
        self.train_mode = train_mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        try:
            frames = load_clip_from_video(
                row["path"],
                clip_len=self.clip_len,
                train_mode=self.train_mode
            )

            clip = []
            for frame in frames:
                x = self.transform(image=frame)["image"]
                clip.append(x)

            clip = torch.stack(clip, dim=0)

        except Exception as e:
            print(f"[WARN] {row['path']} -> {e}")
            blank = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
            clip = []
            for _ in range(self.clip_len):
                x = self.transform(image=blank)["image"]
                clip.append(x)
            clip = torch.stack(clip, dim=0)

        return {
            "clip": clip,
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
            "path": row["path"]
        }

In [12]:
train_ds = PayutchDataset(pay_train_df, transform=train_transform, clip_len=CLIP_LEN, train_mode=True)
val_ds   = PayutchDataset(pay_val_df,   transform=valid_transform, clip_len=CLIP_LEN, train_mode=False)
test_ds  = PayutchDataset(pay_test_df,  transform=valid_transform, clip_len=CLIP_LEN, train_mode=False)

train_labels = pay_train_df["label"].values
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[train_labels]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

PIN_MEMORY = True if DEVICE == "cuda" else False

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print("Class counts:", class_counts)
print("Class weights:", class_weights)

Class counts: [1249  801]
Class weights: [0.00080064 0.00124844]


In [13]:
class AttentionPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.Tanh(),
            nn.Linear(dim // 2, 1)
        )

    def forward(self, x):
        w = self.net(x).squeeze(-1)
        w = torch.softmax(w, dim=1)
        return torch.sum(x * w.unsqueeze(-1), dim=1)

class FallNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            BACKBONE_NAME,
            pretrained=False,
            num_classes=0,
            global_pool="avg"
        )
        feat_dim = self.backbone.num_features

        self.gru = nn.GRU(
            input_size=feat_dim,
            hidden_size=GRU_HIDDEN,
            num_layers=GRU_LAYERS,
            batch_first=True,
            bidirectional=True,
            dropout=0.0
        )

        self.pool = AttentionPool(GRU_HIDDEN * 2)

        self.head = nn.Sequential(
            nn.LayerNorm(GRU_HIDDEN * 2),
            nn.Dropout(DROPOUT),
            nn.Linear(GRU_HIDDEN * 2, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(DROPOUT),
            nn.Linear(128, NUM_CLASSES)
        )

    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)
        feats = self.backbone(x)
        feats = feats.view(b, t, -1)
        seq_out, _ = self.gru(feats)
        pooled = self.pool(seq_out)
        return self.head(pooled)

model = FallNet().to(DEVICE)

state = torch.load(PRETRAINED_PATH, map_location=DEVICE)
model.load_state_dict(state)

print("Loaded checkpoint from:", PRETRAINED_PATH)

Loaded checkpoint from: /kaggle/input/datasets/sadiamostafa/improved/best_payutch_improved_model.pth


In [14]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

def build_optimizer(model):
    backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
    other_params = [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad]

    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": LR_BACKBONE})
    if other_params:
        groups.append({"params": other_params, "lr": LR_HEAD})

    return optim.AdamW(groups, weight_decay=WEIGHT_DECAY)

scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE == "cuda"))

def compute_metrics(y_true, y_pred):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "cm": confusion_matrix(y_true, y_pred)
    }

In [15]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    running_loss = 0.0
    y_true_all, y_pred_all = [], []

    for batch in loader:
        clips = batch["clip"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(clips)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * clips.size(0)
        preds = torch.argmax(logits, dim=1)

        y_true_all.extend(labels.detach().cpu().numpy().tolist())
        y_pred_all.extend(preds.detach().cpu().numpy().tolist())

    m = compute_metrics(y_true_all, y_pred_all)
    m["loss"] = running_loss / len(loader.dataset)
    return m

@torch.no_grad()
def validate_one_epoch(model, loader):
    model.eval()
    running_loss = 0.0
    y_true_all, y_pred_all = [], []

    for batch in loader:
        clips = batch["clip"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(clips)
            loss = criterion(logits, labels)

        running_loss += loss.item() * clips.size(0)
        preds = torch.argmax(logits, dim=1)

        y_true_all.extend(labels.detach().cpu().numpy().tolist())
        y_pred_all.extend(preds.detach().cpu().numpy().tolist())

    m = compute_metrics(y_true_all, y_pred_all)
    m["loss"] = running_loss / len(loader.dataset)
    return m

In [16]:
history = []
best_score = -1
best_path = f"{SAVE_DIR}/best_payutch_final_refined_model.pth"

optimizer = build_optimizer(model)

for epoch in range(1, EPOCHS_TUNE + 1):
    start = time.time()

    train_m = train_one_epoch(model, train_loader, optimizer)
    val_m = validate_one_epoch(model, val_loader)

    score = val_m["f1"] + 0.30 * val_m["recall"]

    history.append({
        "epoch": epoch,
        "train_loss": train_m["loss"],
        "train_f1": train_m["f1"],
        "val_loss": val_m["loss"],
        "val_f1": val_m["f1"],
        "val_recall": val_m["recall"],
        "seconds": time.time() - start
    })

    print(
        f"[Refine {epoch:02d}] "
        f"train_loss={train_m['loss']:.4f} train_f1={train_m['f1']:.4f} | "
        f"val_loss={val_m['loss']:.4f} val_f1={val_m['f1']:.4f} val_recall={val_m['recall']:.4f}"
    )

    if score > best_score:
        best_score = score
        torch.save(model.state_dict(), best_path)
        print("Saved best model")

pd.DataFrame(history).to_csv(f"{SAVE_DIR}/history.csv", index=False)
print("Best score:", best_score)
print("Best model path:", best_path)

[Refine 01] train_loss=0.1537 train_f1=0.9884 | val_loss=0.1575 val_f1=0.9803 val_recall=0.9900
Saved best model
[Refine 02] train_loss=0.1408 train_f1=0.9923 | val_loss=0.1670 val_f1=0.9779 val_recall=0.9900
[Refine 03] train_loss=0.1388 train_f1=0.9942 | val_loss=0.1629 val_f1=0.9754 val_recall=0.9851
[Refine 04] train_loss=0.1344 train_f1=0.9952 | val_loss=0.1692 val_f1=0.9779 val_recall=0.9900
Best score: 1.2773104918755975
Best model path: /kaggle/working/payutch_final_refine/best_payutch_final_refined_model.pth


In [17]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))

val_m = validate_one_epoch(model, val_loader)
test_m = validate_one_epoch(model, test_loader)

print("VAL:")
for k, v in val_m.items():
    if k != "cm":
        print(k, ":", v)
print("VAL CM:\n", val_m["cm"])

print("\nTEST:")
for k, v in test_m.items():
    if k != "cm":
        print(k, ":", v)
print("TEST CM:\n", test_m["cm"])

VAL:
acc : 0.9844054580896686
precision : 0.9707317073170731
recall : 0.9900497512437811
f1 : 0.9802955665024631
loss : 0.15745587835767347
VAL CM:
 [[306   6]
 [  2 199]]

TEST:
acc : 0.9750390015600624
precision : 0.96
recall : 0.975609756097561
f1 : 0.967741935483871
loss : 0.18275630755376146
TEST CM:
 [[385  10]
 [  6 240]]
